# 04_data_performance — Производительность обработки данных для ML-инженера

Этот ноутбук — интенсивная подготовка к собеседованию на ML Engineer с фокусом на **производительность**, **память** и **масштабирование data-пайплайнов**.

> Язык объяснений: **русский**.  
> Язык кода: **Python**.

---

## Как работать с ноутбуком

Для каждой темы вы найдете:
1. Глубокую теорию.
2. Примеры кода.
3. Подробный разбор примеров.
4. Обсуждение производительности.
5. Вопросы в стиле собеседования.
6. Практические мини-задачи.
7. Edge cases (подводные камни).

ML-ориентация: примеры связаны с типичными задачами feature engineering, inference-preprocessing и training-пайплайнов.

## 1) `list` vs `numpy.ndarray`: layout памяти и производительность

### Теория

**Python list** хранит ссылки на объекты, а не сами числа в плотном блоке памяти.  
Это означает:
- дополнительную индирекцию (pointer chasing);
- плохую локальность данных для CPU cache;
- высокий overhead на каждый элемент (объект Python + refcount + тип и т.д.).

**NumPy array** хранит данные в непрерывном (или страйдированном) буфере фиксированного dtype:
- компактное хранение;
- SIMD-оптимизации в C/Fortran-ядре;
- меньше накладных расходов интерпретатора.

Для ML это критично: вектора признаков, матрицы батчей и тензорные преобразования выигрывают от плотного memory layout.

In [ ]:
import sys
import numpy as np

n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n, dtype=np.int64)

list_size = sys.getsizeof(py_list) + sum(sys.getsizeof(x) for x in py_list[:10_000]) * (n / 10_000)
arr_size = np_arr.nbytes

print(f"Оценка памяти list (приближенно): {list_size / (1024**2):.2f} MB")
print(f"Память numpy array (точно по буферу): {arr_size / (1024**2):.2f} MB")
print(f"dtype массива: {np_arr.dtype}")

In [ ]:
import timeit

setup = "import numpy as np; n=1_000_000; py_list=list(range(n)); np_arr=np.arange(n, dtype=np.int64)"
list_stmt = "[x * 2 for x in py_list]"
np_stmt = "np_arr * 2"

list_time = timeit.timeit(list_stmt, setup=setup, number=5)
np_time = timeit.timeit(np_stmt, setup=setup, number=5)

print(f"List comprehension: {list_time:.4f} sec")
print(f"NumPy vectorized:  {np_time:.4f} sec")
print(f"Ускорение NumPy ~ x{list_time/np_time:.1f}")

### Разбор примера

- В первом блоке сравнивается память: `np.ndarray.nbytes` показывает размер data-buffer, тогда как list требует оценку, потому что хранит ссылки и объекты.
- Во втором блоке операция `* 2` для NumPy выполняется в C-цикле, а list comprehension — в Python-цикле.

### Performance discussion

- Чем больше массив и проще операция (арифметика, сравнения), тем больше относительный выигрыш NumPy.
- При очень маленьких массивах overhead вызова NumPy может «съедать» выигрыш.
- Критично выбирать корректный `dtype` (`float32` vs `float64`, `int32` vs `int64`) под требования модели и платформы.

### Вопросы с собеседования

1. Почему `list` почти всегда медленнее `numpy.ndarray` на массовой арифметике?  
2. Что такое contiguous memory и почему это важно для CPU cache?  
3. Когда `float32` предпочтительнее `float64` в ML-пайплайне?

### Мини-задачи

1. Измерьте разницу скорости для `+`, `*`, `np.sqrt` на 10 млн элементов.  
2. Сравните память `float64` vs `float32` и оцените влияние на throughput.  
3. Подберите dtype для one-hot признака (0/1), чтобы минимизировать память.

### Edge cases

- `dtype=object` в NumPy разрушает преимущества векторизации (фактически возвращаемся к Python-объектам).
- Непрерывность памяти может быть нарушена slicing/transpose; иногда нужен `np.ascontiguousarray`.

## 2) Принципы векторизации

### Теория

Векторизация — перенос циклов из Python-уровня в низкоуровневые оптимизированные реализации (обычно C/BLAS/SIMD).  
Ключевая идея: выражать вычисления как операции над массивами, а не как `for` поэлементно.

В ML-практике это:
- массовый препроцессинг признаков;
- нормализация, стандартизация, clipping;
- пакетное вычисление метрик/лоссов.

In [ ]:
import numpy as np

x = np.random.randn(1_000_000).astype(np.float32)

# Невекторизованный стиль (псевдо): for i in range(len(x)): x[i] = ...
# Векторизованный стиль:
zscore = (x - x.mean()) / (x.std() + 1e-8)
clipped = np.clip(zscore, -3, 3)

print(zscore[:5], clipped[:5])

### Разбор примера

- `x.mean()` и `x.std()` считают агрегаты в C.
- Выражение для z-score применяется ко всему массиву без Python-цикла.
- `np.clip` — тоже векторная операция.

### Performance discussion

- Векторизация особенно эффективна на больших батчах.
- Иногда несколько chained-операций создают временные массивы; это расход памяти. Можно снижать overhead через `out=` и in-place операции.

### Вопросы с собеседования

1. Почему Python-loop плох для численных задач?  
2. Что важнее: асимптотика или константы? Приведите пример из NumPy.
3. Какие риски у чрезмерной векторизации с точки зрения памяти?

### Мини-задачи

1. Перепишите поэлементную функцию нормализации в векторном стиле.  
2. Добавьте защиту от деления на ноль для std=0.  
3. Сравните время loop vs vectorized через `timeit`.

### Edge cases

- Векторизация не всегда лучшая для сложной ветвистой логики (много `if/else`), где может понадобиться numba/cython/polars.
- На маленьких массивах эффект может быть незначительным.

## 3) Почему `pandas` часто быстрее Python-циклов

### Теория

`pandas` опирается на NumPy (а в новых версиях местами и Arrow), использует колоночное хранение и векторные операции.

Почему быстрее:
- операции над целыми столбцами вместо `for row in ...`;
- типизированные буферы данных;
- C-уровневые реализации агрегатов/фильтраций/groupby.

ML-контекст: подготовка табличных признаков, агрегации по пользователям/сессиям, feature crosses.

In [ ]:
import pandas as pd
import numpy as np
import timeit

N = 300_000
df = pd.DataFrame({
    'x': np.random.randn(N),
    'y': np.random.randn(N),
})

def python_loop_score(dataframe):
    out = []
    for _, row in dataframe.iterrows():
        out.append(0.7 * row['x'] + 0.3 * row['y'])
    return out

def pandas_vectorized(dataframe):
    return 0.7 * dataframe['x'] + 0.3 * dataframe['y']

loop_time = timeit.timeit(lambda: python_loop_score(df.head(20_000)), number=1)
vec_time = timeit.timeit(lambda: pandas_vectorized(df.head(20_000)), number=10)

print(f"iterrows (20k, 1 run): {loop_time:.4f} sec")
print(f"vectorized (20k, 10 runs): {vec_time:.4f} sec")

### Разбор примера

- `iterrows()` создает Series на каждую строку, что дорого.
- Векторный вариант вычисляет столбец за одну операцию.
- Сравнение сделано на подвыборке для разумного времени выполнения в ноутбуке.

### Performance discussion

- При row-wise логике сначала ищите векторную формулировку (`where`, `cut`, `map`, `merge`, `groupby.transform`).
- Если логика сложная — рассмотрите `itertuples()` (быстрее `iterrows`), numba, либо перенос в SQL/Spark.

### Вопросы с собеседования

1. Почему `iterrows()` медленный?  
2. В чем разница между `apply(axis=1)` и векторными операциями?  
3. Когда разумно уйти из pandas в Spark/Dask/Polars?

### Мини-задачи

1. Перепишите row-wise feature calculation в векторном стиле.  
2. Сравните `iterrows`, `itertuples`, `apply`, vectorized.  
3. Проверьте влияние категориального dtype на память.

### Edge cases

- Смешанные типы в столбце (`object`) сильно замедляют операции.
- Чрезмерное копирование DataFrame (`df = df[...]`) может бить по памяти.

## 4) Broadcasting в NumPy

### Теория

Broadcasting позволяет выполнять операции над массивами разной формы без явного копирования данных, если формы совместимы справа налево:
- размерности равны, или
- одна из размерностей равна 1.

ML-пример: вычитание среднего по признакам из всего батча.

In [ ]:
X = np.random.randn(5, 3).astype(np.float32)  # batch=5, features=3
mean = X.mean(axis=0, keepdims=True)            # shape (1, 3)
X_centered = X - mean                            # broadcast по batch-оси

print("X shape:", X.shape)
print("mean shape:", mean.shape)
print("centered mean ~", X_centered.mean(axis=0))

### Разбор примера

- `mean` имеет форму `(1, 3)` и автоматически «растягивается» до `(5, 3)` при вычитании.
- Физического дублирования `mean` обычно не происходит: это логическая операция на уровне strides/итерации.

### Performance discussion

- Broadcasting часто быстрее и чище, чем `for` по батчам.
- Неверные формы приводят к `ValueError`; полезно явно проверять `.shape`.

### Вопросы с собеседования

1. По каким правилам NumPy проверяет совместимость форм?  
2. Почему broadcasting может быть memory-efficient?  
3. Когда broadcasting может привести к неожиданно большому временному массиву?

### Мини-задачи

1. Нормализуйте матрицу `(N, D)` по каждому признаку (z-score).  
2. Реализуйте pairwise L2 distance между двумя наборами векторов с broadcasting.  
3. Отладьте ошибку несовместимых форм на намеренно неправильном примере.

### Edge cases

- Потенциальные взрывы памяти при неосторожном выражении (например, `(N,1,D) - (1,M,D)` для очень больших N и M).

## 5) Memory efficiency в data pipelines

### Теория

Оптимизация памяти часто важнее micro-оптимизации CPU:
- правильные dtype (`float32`, `int16`, `category`);
- chunk processing вместо загрузки «всё в RAM»;
- удаление временных объектов и избегание лишних копий;
- использование memory-mapped форматов.

В ML это уменьшает OOM-риски при обучении и ускоряет I/O.

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'user_id': np.random.randint(0, 100_000, size=200_000),
    'event_type': np.random.choice(['click', 'view', 'purchase'], size=200_000),
    'value': np.random.randn(200_000),
})

before = df.memory_usage(deep=True).sum() / (1024**2)

opt = df.copy()
opt['user_id'] = opt['user_id'].astype('int32')
opt['event_type'] = opt['event_type'].astype('category')
opt['value'] = opt['value'].astype('float32')

after = opt.memory_usage(deep=True).sum() / (1024**2)

print(f"До оптимизации: {before:.2f} MB")
print(f"После оптимизации: {after:.2f} MB")
print(f"Экономия: {(before-after)/before*100:.1f}%")

### Разбор примера

- `category` кодирует повторяющиеся строки через словарь + integer codes.
- `float32` и `int32` часто достаточны для признаков и уменьшают память в 2 раза относительно 64-битных типов.

### Performance discussion

- Меньше памяти → лучше cache locality → быстрее операции.
- Но downcast может ухудшить численную стабильность (особенно для некоторых статистик/градиентов).

### Вопросы с собеседования

1. Почему `category` полезен для высокоповторяемых строк?  
2. Какие риски у downcasting `float64 -> float32`?  
3. Что лучше для больших массивов: copy-on-write/вьюхи или явные копии?

### Мини-задачи

1. Напишите функцию авто-downcast числовых колонок.  
2. Оцените memory usage до/после для реального CSV.  
3. Проверьте, меняется ли качество модели после downcast.

### Edge cases

- Для колонок с почти уникальными строками `category` может не дать выигрыша.
- Слишком агрессивный downcast может вызвать overflow/underflow.

## 6) Генераторы для больших датасетов

### Теория

Генераторы (`yield`) отдают элементы по одному и не хранят весь набор в памяти.  
Это особенно полезно для:
- потокового чтения логов;
- online feature extraction;
- батчевой подачи данных в обучение.

In [ ]:
def batch_generator(iterable, batch_size=1024):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

# пример: потоковая обработка синтетических записей
records = ({'x': i, 'y': i % 7} for i in range(10_000))

total = 0
for batch in batch_generator(records, batch_size=512):
    total += sum(r['x'] for r in batch)

print("Сумма x:", total)

### Разбор примера

- Данные не материализуются целиком как список.
- Мы контролируем пик памяти через `batch_size`.

### Performance discussion

- Генераторы уменьшают memory footprint, но могут быть чуть медленнее на элемент из-за частых переходов между Python frame.
- В реальных ETL обычно выгоднее снижение памяти и устойчивость к OOM.

### Вопросы с собеседования

1. Чем генератор отличается от списка по памяти и жизненному циклу?  
2. Что происходит с генератором после полного прохода?  
3. Как организовать backpressure в потоковом пайплайне?

### Мини-задачи

1. Реализуйте генератор чтения CSV по чанкам.  
2. Постройте генератор признаков для текстовых данных.  
3. Сравните пик памяти list vs generator.

### Edge cases

- Генератор одноразовый: повторный проход потребует пересоздания.
- Ошибка в середине стрима может привести к потере контекста; нужна стратегия checkpoint/retry.

## 7) Lazy evaluation (ленивые вычисления)

### Теория

Ленивые вычисления откладывают выполнение до момента, когда результат действительно нужен.  
Примеры в Python: генераторы, `map`, `filter`, итераторы файлов.  
В экосистеме данных: Dask/Spark строят граф вычислений и выполняют его позже.

ML-идея: отложенная подготовка признаков до батч-инференса, чтобы не хранить всё заранее.

In [ ]:
nums = range(1, 1_000_001)

lazy_pipeline = map(lambda x: x * x, nums)
lazy_pipeline = filter(lambda x: x % 3 == 0, lazy_pipeline)

# До этого момента вычисления почти не выполнялись.
first_five = [next(lazy_pipeline) for _ in range(5)]
print(first_five)

### Разбор примера

- `map` и `filter` возвращают итераторы.
- Вычисление происходит только при `next()` или итерации.

### Performance discussion

- Lazy-пайплайн экономит память и может ускорить время до первого результата.
- Полная пропускная способность зависит от стоимости функции и накладных расходов Python.

### Вопросы с собеседования

1. Чем lazy evaluation отличается от eager?  
2. Когда важнее latency до первого батча, а когда total throughput?  
3. Как отладить ошибку в длинной цепочке ленивых итераторов?

### Мини-задачи

1. Постройте lazy-конвейер очистки текстов (lower, strip, фильтрация).  
2. Добавьте логирование каждого N-го элемента без материализации всего потока.  
3. Сравните memory usage eager vs lazy.

### Edge cases

- Одноразовость итераторов: после прохода данные «исчезают».
- Ошибки могут проявляться поздно (в момент потребления), усложняя дебаг.

## 8) `map` / `filter` / `reduce` в подготовке данных

### Теория

Это функциональные примитивы для трансформации последовательностей:
- `map`: преобразование каждого элемента;
- `filter`: отбор по условию;
- `reduce`: свертка к одному значению.

В ML полезны для легких потоковых преобразований перед векторизацией.

In [ ]:
from functools import reduce

samples = [
    {'feature': 0.1, 'label': 1},
    {'feature': -0.3, 'label': 0},
    {'feature': 0.8, 'label': 1},
]

transformed = map(lambda s: {'feature': s['feature'] * 10, 'label': s['label']}, samples)
positives = filter(lambda s: s['feature'] > 0, transformed)
feature_sum = reduce(lambda acc, s: acc + s['feature'], positives, 0.0)

print("Сумма положительных transformed features:", feature_sum)

### Разбор примера

- Мы сделали последовательный pipeline без промежуточных списков.
- `reduce` применим для агрегатов, но в численных задачах обычно предпочтительнее NumPy/pandas-агрегации.

### Performance discussion

- `map/filter` с Python-лямбдами не всегда быстрее list comprehension.
- Их плюс — композиционность и laziness (в Py3).

### Вопросы с собеседования

1. Когда `reduce` ухудшает читаемость?  
2. Что выбрать: `map/filter` или comprehension?  
3. Почему для числовой математики лучше NumPy?

### Мини-задачи

1. Перепишите pipeline через list comprehension и сравните читаемость/скорость.  
2. Добавьте обработку пропусков (`None`) в map/filter конвейере.  
3. Реализуйте reduce для подсчета confusion matrix из стрима предсказаний.

### Edge cases

- Лямбды с побочными эффектами затрудняют отладку.
- `reduce` без стартового значения может падать на пустых входах.

## 9) Базовый profiling: `timeit` и `cProfile`

### Теория

Профилирование отвечает на два вопроса:
1. **Где время?** (hotspots)
2. **Почему медленно?** (алгоритм, I/O, аллокации, Python overhead)

`timeit` — точечный микробенчмарк.  
`cProfile` — профилирование вызовов функций и cumulative time.

In [ ]:
import cProfile
import pstats
import io
import numpy as np

def slow_pipeline(n=200_000):
    data = list(range(n))
    data = [x * 2 for x in data]
    data = [x for x in data if x % 3 == 0]
    return sum(data)

pr = cProfile.Profile()
pr.enable()
res = slow_pipeline(150_000)
pr.disable()

s = io.StringIO()
ps = pstats.Stats(pr, stream=s).sort_stats('cumtime')
ps.print_stats(10)

print("Результат:", res)
print(s.getvalue())

### Разбор примера

- `cProfile` показывает функции, где накоплено больше всего времени (`cumtime`).
- Это помогает решать, что оптимизировать первым.

### Performance discussion

- Сначала профилируем, потом оптимизируем — избегаем «оптимизации не того места».
- Для стабильных измерений запускайте несколько прогонов и прогревайте окружение.

### Вопросы с собеседования

1. В чем разница `tottime` и `cumtime`?  
2. Почему microbenchmark может врать о production-перфомансе?  
3. Как сравнить два варианта функции корректно?

### Мини-задачи

1. Найдите hotspot в собственном preprocessing-коде.  
2. Сравните 3 реализации одной задачи через `timeit`.  
3. Постройте таблицу speedup относительно baseline.

### Edge cases

- Профилирование может искажать время (observer effect).
- I/O-bound и CPU-bound сценарии требуют разных инструментов анализа.

## 10) Безопасная работа с большими файлами

### Теория

При работе с большими CSV/JSONL важно:
- читать чанками (`chunksize`);
- валидировать схему и типы;
- избегать полной загрузки в память;
- аккуратно обрабатывать ошибки формата.

ML-контекст: оффлайн фичегенерация и построение train dataset из логов.

In [ ]:
import pandas as pd
from pathlib import Path

# Демонстрационный файл
path = Path('demo_large.csv')
if not path.exists():
    pd.DataFrame({
        'user_id': range(100_000),
        'score': np.random.randn(100_000),
    }).to_csv(path, index=False)

running_sum = 0.0
rows = 0
for chunk in pd.read_csv(path, chunksize=20_000, dtype={'user_id': 'int32', 'score': 'float32'}):
    running_sum += chunk['score'].sum()
    rows += len(chunk)

print(f"Строк обработано: {rows}")
print(f"Сумма score: {running_sum:.4f}")

### Разбор примера

- `chunksize` ограничивает объем данных в памяти.
- Явный `dtype` предотвращает неоптимальные авто-выводы типов.
- Такой паттерн устойчивее для production ETL.

### Performance discussion

- Размер чанка влияет на баланс I/O и CPU: слишком маленький — overhead, слишком большой — риск OOM.
- Сжатые форматы (parquet) обычно эффективнее CSV по скорости и памяти.

### Вопросы с собеседования

1. Как подобрать `chunksize`?  
2. Почему parquet часто предпочтительнее CSV для аналитики?  
3. Как обеспечить идемпотентность при падении обработки на середине файла?

### Мини-задачи

1. Добавьте валидацию диапазона `score` и лог ошибок.  
2. Сохраните агрегаты по чанкам в отдельный parquet.  
3. Реализуйте restart-safe обработку с checkpoint по offset.

### Edge cases

- Поврежденные строки, неожиданные разделители, encoding-ошибки.
- Чтение сетевых файлов требует retry/backoff и таймаутов.

## 11) Базовый memory profiling

### Теория

Время — не единственный ресурс. Важны:
- peak RSS процесса;
- количество аллокаций;
- временные копии массивов/DataFrame.

Для быстрой диагностики в стандартной библиотеке есть `tracemalloc`.

In [ ]:
import tracemalloc

def memory_heavy_op(n=1_000_000):
    a = [float(i) for i in range(n)]
    b = [x * 1.1 for x in a]
    return sum(b)

tracemalloc.start()
_ = memory_heavy_op(300_000)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Текущая память: {current / (1024**2):.2f} MB")
print(f"Пиковая память:  {peak / (1024**2):.2f} MB")

### Разбор примера

- Функция создает два больших списка => заметный пик памяти.
- `tracemalloc` помогает увидеть, что оптимизировать (например, перейти на NumPy или генераторы).

### Performance discussion

- Пиковая память часто ограничивает масштаб батча сильнее, чем CPU.
- Для production полезно дополнительно мониторить RSS/контейнерные лимиты.

### Вопросы с собеседования

1. Чем `current` отличается от `peak` memory?  
2. Как уменьшить peak memory без потери точности?  
3. Почему временные объекты опасны в feature engineering?

### Мини-задачи

1. Перепишите `memory_heavy_op` через генераторы и сравните peak memory.  
2. Сравните list vs NumPy версию функции.  
3. Добавьте контроль памяти в CI smoke-тест.

### Edge cases

- `tracemalloc` не показывает всю внешнюю память C-расширений идеально.
- GC-паузы и фрагментация могут скрывать реальную картину.

## 12) Анализ сложности в data processing

### Теория

Асимптотика (Big-O) помогает понять, как pipeline масштабируется с ростом данных:
- O(N) — линейный проход;
- O(N log N) — сортировки, некоторые join/merge сценарии;
- O(N²) — попарные сравнения (опасно для больших N).

В ML-пайплайнах критично избегать скрытых O(N²), особенно на этапах feature matching/duplicate detection.

In [ ]:
import time
import numpy as np

def linear_sum(arr):
    s = 0
    for x in arr:
        s += x
    return s

def quadratic_pairs(arr):
    cnt = 0
    for i in range(len(arr)):
        for j in range(len(arr)):
            if arr[i] < arr[j]:
                cnt += 1
    return cnt

for n in [300, 600, 1200]:
    arr = np.random.randint(0, 1000, size=n)

    t0 = time.perf_counter()
    linear_sum(arr)
    t1 = time.perf_counter()

    t2 = time.perf_counter()
    quadratic_pairs(arr[:200])  # ограничиваем, чтобы не перегружать ноутбук
    t3 = time.perf_counter()

    print(f"n={n:4d} | O(N): {(t1-t0)*1e3:.2f} ms | O(N^2) на 200: {(t3-t2)*1e3:.2f} ms")

### Разбор примера

- Линейный проход масштабируется заметно мягче.
- Квадратичная часть быстро становится дорогой, поэтому ограничена `arr[:200]`.

### Performance discussion

- Даже «быстрый» язык/библиотека не спасет от плохой асимптотики.
- Первый шаг оптимизации — уменьшить сложность алгоритма, второй — оптимизировать реализацию.

### Вопросы с собеседования

1. Пример скрытого O(N²) в pandas/SQL пайплайне.  
2. Почему hash-based join обычно лучше nested loop join?  
3. Как объяснить бизнесу компромисс между точностью и сложностью?

### Мини-задачи

1. Найдите O(N²) участок в своем коде и предложите O(N log N) или O(N) замену.  
2. Оцените time/memory complexity для feature deduplication.  
3. Подготовьте краткий perf-review: bottleneck, метрика, план улучшения.

### Edge cases

- Асимптотика может быть одинаковой, но константы — радикально разными.
- Реальные данные с перекосами распределений ломают «средние» ожидания.

## Финальный блок: чек-лист перед ML-собеседованием

- Умеете объяснить разницу между Python-объектами и плотным числовым буфером.
- Умеете переписывать циклы в векторные операции.
- Понимаете trade-off между CPU временем и памятью.
- Умеете профилировать (`timeit`, `cProfile`, базовый memory profiling).
- Умеете безопасно обрабатывать большие файлы и строить потоковые пайплайны.
- Можете оценить алгоритмическую сложность и масштабируемость решения.

> Рекомендуется: прогнать все ячейки, сохранить собственные замеры, и для каждой темы подготовить устный ответ на 1–2 минуты в стиле интервью.